# ⚽ FIFA Player Scouting & Performance Analytics
### Data Cleaning, Feature Engineering & Interactive Scouting Intelligence

**Author:** [Your Name]
**Contact Number:** [Your Phone Number]
**Email:** [Your Email Address]
**GitHub:** [Your GitHub Profile URL]
**LinkedIn:** [Your LinkedIn Profile URL]

---
## 📌 Project Overview
This Jupyter Notebook processes raw FIFA player data (`players_22.csv`) to perform data cleaning, feature engineering, and calculate key scouting metrics such as **Growth Potential**, **Value for Money Score**, and **Hidden Gem Flags** for recruitment and talent analysis.

### Step 1: Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

# Set plot styling
sns.set_theme(style="darkgrid")
print("Libraries loaded successfully!")

### Step 2: Load Raw Dataset (`players_22.csv`)

In [ ]:
raw_df = pd.read_csv("players_22.csv")
print(f"Raw dataset loaded! Shape: {raw_df.shape}")
raw_df.head()

### Step 3: Data Cleaning & Missing Value Handling

In [ ]:
# Fill missing club and country names
raw_df["club_name"] = raw_df["club_name"].fillna("Free Agent")
raw_df["nationality_name"] = raw_df["nationality_name"].fillna("Unknown")

# Extract Primary Position
raw_df["primary_position"] = raw_df["player_positions"].astype(str).apply(lambda x: x.split(",")[0].strip())
print("Data cleaning completed successfully!")

### Step 4: Position Grouping & Age Bucketing

In [ ]:
def get_position_group(pos):
    pos = str(pos).upper()
    if pos == "GK":
        return "Goalkeeper"
    elif pos in ["CB", "LB", "RB", "LWB", "RWB"]:
        return "Defender"
    elif pos in ["CDM", "CM", "CAM", "LM", "RM"]:
        return "Midfielder"
    else:
        return "Attacker"

raw_df["position_group"] = raw_df["primary_position"].apply(get_position_group)

def get_age_bucket(age):
    if age <= 21:
        return "U21"
    elif age <= 25:
        return "21-25"
    elif age <= 30:
        return "26-30"
    else:
        return "30+"

raw_df["age_bucket"] = raw_df["age"].apply(get_age_bucket)
print("Position Group Counts:")
print(raw_df["position_group"].value_counts())

### Step 5: Feature Engineering - Scouting Metrics

In [ ]:
# Growth Potential
raw_df["growth_potential"] = raw_df["potential"] - raw_df["overall"]

# Market Value in Millions
raw_df["value_million"] = raw_df["value_eur"] / 1000000.0

# Value for Money Score
raw_df["value_for_money"] = np.where(raw_df["value_million"] > 0, raw_df["overall"] / raw_df["value_million"], 0).round(2)

# Hidden Gem Flag (Age <= 23 & Growth >= 8)
raw_df["hidden_gem_flag"] = np.where(
    (raw_df["growth_potential"] >= 8) & (raw_df["age"] <= 23),
    "Hidden Gem",
    "Regular Target"
)

print(f"Total Hidden Gems Identified: {(raw_df['hidden_gem_flag'] == 'Hidden Gem').sum()}")

### Step 6: Exploratory Data Visualization

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=raw_df, 
    x="overall", 
    y="potential", 
    hue="hidden_gem_flag", 
    palette={"Hidden Gem": "#F59E0B", "Regular Target": "#3B82F6"},
    alpha=0.85,
    s=75
)
plt.title("FIFA Player Analytics: Overall Rating vs Potential", fontsize=14, fontweight="bold")
plt.xlabel("Current Overall Rating")
plt.ylabel("Potential Rating")
plt.show()

### Step 7: Export Clean Datasets for Power BI & Web Dashboard

In [ ]:
# Export CSV for Power BI
raw_df.to_csv("cleaned_players.csv", index=False)

# Export JSON for Web Dashboard
records = raw_df.to_dict(orient="records")
with open("players_data.json", "w", encoding="utf-8") as f:
    json.dump(records, f, indent=2)

print("Export completed successfully!")